##### Parameter Maps (MERIT)

This notebook loads the geometry predictions NetCDF produced by
`geometry_predictor.py` and visualizes the spatial distribution of learned
routing parameters and predicted channel geometry on CONUS catchment maps.

**To generate the dataset**, run the Hydra script first:

```bash
python scripts/geometry_predictor.py --config-name=merit_geometry_config
```

Plottable variables from the NetCDF:
- **KAN parameters**: `n`, `q_spatial`, `p_spatial`
- **Channel geometry stats**: `{depth,top_width,discharge}_{min,max,median,mean}`
- **Slope**: `slope`

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
EXAMPLE_DIR = Path(".")
DATA_DIR = EXAMPLE_DIR / ".." / ".." / "data"

# Load geometry predictions from the script output
ds = xr.open_dataset(EXAMPLE_DIR / "merit_geometry_predictions.nc")
comids = ds["COMID"].values

print(f"Loaded {len(comids):,} CONUS reaches")
print(f"Water year: {ds.attrs['water_year']}")
print(f"Variables: {list(ds.data_vars)}")

In [ ]:
# Load catchment polygons and join geometry predictions
catchment_shp = DATA_DIR / "merit" / "cat_pfaf_7_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
gdf = gpd.read_file(catchment_shp).set_index("COMID")

# Subset NetCDF to COMIDs present in the shapefile
shared_comids = np.intersect1d(gdf.index.values, comids)
ds_subset = ds.sel(COMID=shared_comids)

for var in ds_subset.data_vars:
    gdf.loc[shared_comids, var] = ds_subset[var].values

gdf = gdf.set_crs(epsg=4326)
print(f"Joined {len(shared_comids):,} catchment polygons with {len(list(ds_subset.data_vars))} variables")

In [ ]:
def param_plot(
    gdf: gpd.GeoDataFrame,
    var: str,
    save_name: Path,
    cmap: str = "plasma",
    unit_label: str | None = None,
    title: str | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    ascending: bool = False,
    dpi: int = 100,
) -> tuple[Figure, Axes]:
    """
    Create a parameter plot for geospatial data with a basemap and colorbar.

    Parameters
    ----------
    gdf : gpd.GeoDataFrame
        GeoDataFrame containing the data to plot.
    var : str
        Column name to visualize.
    save_name : str
        Filename for saving the plot.
    cmap : str, default 'plasma'
        Colormap name for the plot.
    unit_label : str, optional
        Unit label for the colorbar.
    title : str, optional
        Title for the plot.
    vmin : float, optional
        Minimum value for color scaling. If None, uses data minimum.
    vmax : float, optional
        Maximum value for color scaling. If None, uses data maximum.
    ascending : bool, default False
        Whether to sort data in ascending order.
    dpi : int, default 100
        DPI for the figure display.

    Returns
    -------
    tuple of (matplotlib.figure.Figure, matplotlib.axes.Axes)
        Figure and axes objects for further customization.

    """
    if var not in gdf.columns:
        raise KeyError(f"Column '{var}' not found in GeoDataFrame")

    fig, ax = plt.subplots(figsize=(7, 4), dpi=dpi)

    gdf_clean = gdf.dropna(subset=[var])
    if gdf_clean.empty:
        raise ValueError(f"No valid data found for variable '{var}' after dropping NaN values")

    gdf_clean = gdf_clean.sort_values(by=var, ascending=ascending)
    data = gdf_clean[var].values

    if vmin is None:
        vmin = np.min(data)
    if vmax is None:
        vmax = np.nanmax(data)

    gdf_clean.plot(
        ax=ax,
        column=var,
        cmap=cmap,
        linewidth=0.3,
        vmin=vmin,
        vmax=vmax,
        zorder=1,
    )

    cx.add_basemap(
        ax,
        crs=gdf_clean.crs,
        source=cx.providers.CartoDB.Positron,
        alpha=0.6,
        zorder=0,
        attribution=False,
    )

    # CONUS bounds
    ax.set_xlim(-125, -66)
    ax.set_ylim(24, 53)
    ax.set_xticks([])
    ax.set_yticks([])

    if title is not None:
        ax.set_title(title, fontsize=14)

    # Colorbar legend
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.1)
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array([])
    sm.set_clim(vmin, vmax)
    cbar = fig.colorbar(sm, cax=cax)

    label_text = var
    if unit_label:
        label_text = f"{var} ({unit_label})"
    cbar.set_label(label_text)

    cbar.formatter.set_powerlimits((-2, 2))
    cbar.update_ticks()

    plt.tight_layout()
    plt.savefig(save_name, dpi=dpi, bbox_inches="tight")

    return fig, ax

In [ ]:
# Plot configuration for geometry prediction variables
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(exist_ok=True)

PLOT_CONFIGS = {
    "n": {
        "title": "Manning's Roughness",
        "unit_label": "m⁻¹/³s",
        "cmap": "plasma_r",
        "vmax": 0.2,
        "ascending": True,
    },
    "q_spatial": {
        "title": "Width-Depth Exponent (q)",
        "unit_label": "–",
        "cmap": "viridis",
        "ascending": True,
    },
    "p_spatial": {
        "title": "Width Coefficient (p)",
        "unit_label": "–",
        "cmap": "viridis",
        "ascending": True,
    },
    "depth_median": {
        "title": "Median Channel Depth",
        "unit_label": "m",
        "cmap": "Blues",
        "ascending": True,
    },
    "top_width_median": {
        "title": "Median Top Width",
        "unit_label": "m",
        "cmap": "Blues",
        "ascending": True,
    },
}

In [ ]:
for var, cfg in PLOT_CONFIGS.items():
    print(f"Plotting {var}...")
    param_plot(gdf, var, PLOT_DIR / f"{var}.png", dpi=300, **cfg)
    plt.show()